# TIGRFAM

In [ ]:
import gzip, requests
import pandas as pd

In [44]:
found_results = pd.read_csv("../data/example_results_tigr.tsv", sep="\t")

In [23]:
url = "https://ftp.ncbi.nlm.nih.gov/hmm/TIGRFAMs/release_15.0"

with requests.get(f"{url}/TIGRFAMS_ROLE_LINK") as r:
    with open("../data/tigrfams_role_link.txt", "wb") as f:
        f.write(r.content)

with requests.get(f"{url}/TIGR_ROLE_NAMES") as r:
    with open("../data/tigr_role_names.txt", "wb") as f:
        f.write(r.content)

In [25]:
tigrfam_role_link = pd.read_csv("../data/tigrfams_role_link.txt", sep=r"\s+", header=None,
                   names=["tigr_acc", "role_id"], dtype=str, engine="python")

In [32]:
tigr_role_names = pd.read_csv("../data/tigr_role_names.txt", sep="\t", header=None,
names=["role_id_name", "role_id", "type", "description"], dtype=str, engine="python").drop(columns="role_id_name")

In [34]:
tigrfam_role_link

,tigr_acc,role_id
0,TIGR00001,158
1,TIGR00002,158
2,TIGR00003,145
3,TIGR00004,92
4,TIGR00005,168
...,...,...
3267,TIGR04567,135
3268,TIGR04568,168
3269,TIGR04569,168
3270,TIGR04570,157


Combine all information:

In [37]:
results_with_roles = found_results.merge(tigrfam_role_link, how="left", on="tigr_acc").merge(tigr_role_names, how="left", on="role_id")

In [38]:
results_with_roles

,wp_id,tigr_acc,role_id,type,description
0,WP_000745871.1,NF033845,NaN,NaN,NaN
1,WP_001006454.1,TIGR01979,76,mainrole:,"Biosynthesis of cofactors, prosthetic groups, ..."
2,WP_001006454.1,TIGR01979,76,sub1role:,Other
3,WP_000250824.1,TIGR01825,NaN,NaN,NaN
4,WP_000808845.1,TIGR01327,75,mainrole:,Amino acid biosynthesis
5,WP_000808845.1,TIGR01327,75,sub1role:,Serine family
6,WP_001140234.1,TIGR00719,109,mainrole:,Energy metabolism
7,WP_001140234.1,TIGR00719,109,sub1role:,Amino acids and amines
8,WP_001140234.1,TIGR00719,116,mainrole:,Energy metabolism
9,WP_001140234.1,TIGR00719,116,sub1role:,Glycolysis/gluconeogenesis


## Add this to the example data

In [39]:
test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")

/var/folders/z3/xb62blvs5xg7hy35ktdz1mdw0000gn/T/ipykernel_65822/911866719.py:1: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  test_data = pd.ExcelFile("../../Example_Staphylococcus_aureus_Newman_TaxId_426430.xlsx").parse("Tabelle1")


In [40]:
test_data = test_data.assign(wp_id=test_data["Majority protein IDs"].str.extract(r"(WP_[\d.]+)", expand=False))

In [41]:
test_data = test_data.merge(results_with_roles.dropna(subset="description"), how="left", on="wp_id")

In [42]:
test_data

,Majority protein IDs,NWMN neu,NWMN alt,gen symbol,Annotation,TIGRFAM Sub Role,The SEED,wp_id,tigr_acc,role_id,type,description
0,WP_000942300.1|NWMN_RS07340|NWMN_1303|,NWMN_RS07340,NWMN_1303,NWMN_1303,ATP-binding cassette domain-containing protei...,0,NaN,WP_000942300.1,NaN,NaN,NaN,NaN
1,WP_000602071.1|NWMN_RS11270|NWMN_1954|,NWMN_RS11270,NWMN_1954,NWMN_1954,ABC-F type ribosomal protection protein[Staph...,0,NaN,WP_000602071.1,NaN,NaN,NaN,NaN
2,WP_000524836.1|NWMN_RS12920|NWMN_2242|,NWMN_RS12920,NWMN_2242,NWMN_2242,ABC transporter ATP-binding protein[Staphyloc...,Other,NaN,WP_000524836.1,NaN,NaN,NaN,NaN
3,WP_001229077.1|NWMN_RS13625|NWMN_2364|,NWMN_RS13625,NWMN_2364,opp1A,staphylopine-dependent metal ABC transporter ...,Cations and iron carrying compounds,"""ABC transporter dipeptide (TC 3.A.1.5.2)""",WP_001229077.1,NaN,NaN,NaN,NaN
4,WP_001172340.1|NWMN_RS14380|NWMN_2506|,NWMN_RS14380,NWMN_2506,NWMN_2506,AMP-binding protein[Staphylococcus aureus sub...,0,"AMP-(fatty) acid ligases, YtcI homolog""",WP_001172340.1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
199,WP_001108479.1|NWMN_RS11550|NWMN_1996|,NWMN_RS11550,NWMN_1996,thiM,hydroxyethylthiazole kinase[Staphylococcus au...,Thiamine,"""Thiamin biosynthesis""",WP_001108479.1,NaN,NaN,NaN,NaN
200,WP_000396077.1|NWMN_RS11560|NWMN_1998|,NWMN_RS11560,NWMN_1998,NWMN_1998,thiaminase II[Staphylococcus aureus subsp. au...,0,"""Thiamin biosynthesis""",WP_000396077.1,NaN,NaN,NaN,NaN
201,WP_000872653.1|NWMN_RS09020|NWMN_1609|,NWMN_RS09020,NWMN_1609,thiI,tRNA uracil 4-sulfurtransferase ThiI[Staphylo...,Thiamine; tRNA and rRNA base modification,"""Thiamin biosynthesis""",WP_000872653.1,NaN,NaN,NaN,NaN
202,WP_000594959.1|NWMN_RS11555|NWMN_1997|,NWMN_RS11555,NWMN_1997,thiD,bifunctional hydroxymethylpyrimidine kinase/p...,Thiamine,"""Thiamin biosynthesis""",WP_000594959.1,NaN,NaN,NaN,NaN


In [43]:
test_data.to_excel("../data/example_s_aureus_tigrfam.xlsx", index=False)